In [ ]:
# SVM + MLP + LOGISTIC

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from google.colab import files

# LOAD DATA
train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")

TARGET = "spend_category"
ID_COL = "trip_id"

print(train.shape, test.shape)

# CLEAN TARGET
train[TARGET] = pd.to_numeric(train[TARGET], errors="coerce")
train = train.dropna(subset=[TARGET]).reset_index(drop=True)
y = train[TARGET].astype(int)

# FEATURES
if ID_COL in train.columns:
    train = train.drop(columns=[ID_COL])
if ID_COL in test.columns:
    test_ids = test[ID_COL].copy()
    test = test.drop(columns=[ID_COL])

X = train.drop(columns=[TARGET])
X_test = test.copy()

categorical_cols = X.select_dtypes(include="object").columns.tolist()
numeric_cols     = X.select_dtypes(exclude="object").columns.tolist()

def target_encode(col, X, y, X_test):
    means = X.groupby(col)[y.name].mean()
    return X[col].map(means), X_test[col].map(means).fillna(means.mean())

for col in categorical_cols:
    X[col], X_test[col] = target_encode(col, train, y, test)

# After encoding, convert everything to numeric
X = X.apply(pd.to_numeric, errors="coerce")
X_test = X_test.apply(pd.to_numeric, errors="coerce")

# Impute any remaining missing values
X = X.fillna(X.mean())
X_test = X_test.fillna(X.mean())


# SCALING
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y, test_size=0.15, stratify=y, random_state=42
)

# SVM (Linear)
svm = LinearSVC(C=3.0)
svm.fit(X_train, y_train)
svm_train_pred = svm.decision_function(X_train)
svm_val_pred   = svm.decision_function(X_val)
svm_test_pred  = svm.decision_function(X_test_scaled)

# NEURAL NETWORK (MLP)
mlp = MLPClassifier(
    hidden_layer_sizes=(256,128),
    learning_rate_init=0.001,
    alpha=0.0005,
    max_iter=350,
    early_stopping=True,
    random_state=42
)
mlp.fit(X_train, y_train)
mlp_val_pred  = mlp.predict_proba(X_val)
mlp_test_pred = mlp.predict_proba(X_test_scaled)

meta_train = np.hstack([
    svm_val_pred,
    mlp_val_pred
])

meta_test = np.hstack([
    svm_test_pred,
    mlp_test_pred
])

# Logistic Regression
meta_clf = LogisticRegression(max_iter=300, multi_class="multinomial")
meta_clf.fit(meta_train, y_val)

print("Validation accuracy:", accuracy_score(y_val, meta_clf.predict(meta_train)))

final_preds = meta_clf.predict(meta_test)

sub = pd.DataFrame({
    ID_COL: test_ids,
    "category": final_preds
})
sub.to_csv("svm_nn_stacked.csv", index=False)
print("Saved svm_nn_stacked.csv")

files.download("svm_nn_stacked.csv")
